In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import json

In [ ]:
path = kagglehub.dataset_download(
                            "tmdb/tmdb-movie-metadata",
                            output_dir = '../data')
movies_path =  path + "/tmdb_5000_movies.csv"
credits_path = path + "/tmdb_5000_credits.csv"

In [ ]:
movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

In [ ]:
movies_full = pd.merge(left = movies, right = credits, left_on = 'id', right_on = 'movie_id').reset_index(drop = True)

In [ ]:
movies_df = movies_full[['title_x', 'overview', 'keywords', 'genres', 'cast', 'crew']]
movies_df = movies_df.rename({'title_x': 'title'}, axis = 1)

In [ ]:
indices = (
            movies_df['title']
            .reset_index()
            .set_index('title')
           )
indices

In [115]:
movies_df.head()

,title,overview,keywords,genres,cast,crew,top_keywords,top_cast,director,top_genres
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{'id': 1463, 'name': 'culture clash'}, {'id':...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'cast_id': 242, 'character': 'Jake Sully', '...","[{'credit_id': '52fe48009251416c750aca23', 'de...","[3d, alien, future, soldier, battle, space]","[samworthington, zoesaldana, sigourneyweaver, ...",[jamescameron],"[Action, Adventure, Science_Fiction, Fantasy]"
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'cast_id': 4, 'character': 'Captain Jack Spa...","[{'credit_id': '52fe4232c3a36847f800b579', 'de...","[aftercreditsstinger, love_of_one's_life, ship...","[johnnydepp, orlandobloom, keiraknightley, ste...",[goreverbinski],"[Action, Adventure, Fantasy]"
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{'id': 470, 'name': 'spy'}, {'id': 818, 'name...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'cast_id': 1, 'character': 'James Bond', 'cr...","[{'credit_id': '54805967c3a36829b5002c41', 'de...","[based_on_novel, sequel, spy, secret_agent, br...","[danielcraig, christophwaltz, léaseydoux, ralp...",[sammendes],"[Action, Adventure, Crime]"
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{'id': 849, 'name': 'dc comics'}, {'id': 853,...","[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'cast_id': 2, 'character': 'Bruce Wayne / Ba...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de...","[superhero, secret_identity, terrorist, imax, ...","[christianbale, michaelcaine, garyoldman, anne...",[christophernolan],"[Drama, Thriller, Action, Crime]"
4,John Carter,"John Carter is a war-weary, former military ca...","[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'cast_id': 5, 'character': 'John Carter', 'c...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de...","[based_on_novel, 3d, alien, escape, princess, ...","[taylorkitsch, lynncollins, samanthamorton, wi...",[andrewstanton],"[Action, Adventure, Science_Fiction]"


In [ ]:
from ast import literal_eval

features = ['keywords', 'genres', 'cast', 'crew']

for feature in features:
    movies_df[feature] = movies_df[feature].apply(literal_eval)

In [ ]:
movies_df.keywords

In [ ]:
keyword_counts = {}
genre_counts = {}

for movie in movies_df['keywords']:
    for kw_data in movie:
        keyword = kw_data['name']
        keyword_counts[keyword] = keyword_counts.get(keyword, 0) + 1

for movie in movies_df['genres']:
    for genre_data in movie:
        genre = genre_data['name']
        genre_counts[genre] = genre_counts.get(genre, 0) + 1

In [ ]:
movies_df.loc[0, 'keywords']

In [ ]:
keyword_counts['culture clash']

In [99]:
def top_keywords(row, length = 3):
    kw_counts = {kw['name'].replace(' ', '_'): keyword_counts[kw['name']] for kw in row}
    kw_counts = sorted(kw_counts, key = lambda kw: kw_counts[kw], reverse = True)
    kw = [keyword for keyword in kw_counts]

    if len(kw) > length:
        return kw[:length]
    else:
        return kw
    
def top_genres(row, length, all = True):
    gen_counts = {genre['name'].replace(' ', '_'): genre_counts[genre['name']] for genre in row}
    gen_counts = sorted(gen_counts, key = lambda gen: gen_counts[gen], reverse=True)
    genres = [genre for genre in gen_counts]

    if all:
        return genres
    
    if len(genres) > length:
        return genres[:length]
    else:
        return genres
    
    
def top_cast(row, length):
    cast = [actor['name'].lower().replace(' ', '') for actor in row]
    cast = cast[:length]
    return cast

def director(row):
    director = [crew['name'].lower().replace(' ', '') for crew in row if crew['job'] == 'Director']
    return director

In [100]:
movies_df['top_keywords'] = movies_df['keywords'].apply(top_keywords, length = 6)
movies_df['top_cast'] = movies_df['cast'].apply(top_cast, length = 5)
movies_df['director'] = movies_df['crew'].apply(director)
movies_df['top_genres'] = movies_df['genres'].apply(top_genres, length = 3, all = True)
movies_df.rename({'title_x': 'title'}, axis = 1, inplace=True)

In [101]:
movies_df.loc[(movies_df.title == 'Iron Man'), 'top_keywords'].to_list()

[['aftercreditsstinger',
  'superhero',
  'based_on_comic_book',
  'marvel_comic',
  'marvel_cinematic_universe',
  'middle_east']]

In [102]:
def create_soup(x):
    soup = x['top_keywords'] + x['top_cast'] + x['director'] + x['top_genres']
    return ' '.join(soup)

soup_df = pd.DataFrame(movies_df['title'])
soup_df['soup'] = movies_df.apply(create_soup, axis=1)
soup_df = soup_df.set_index('title')

In [103]:
soup_df.loc['Iron Man','soup']

'aftercreditsstinger superhero based_on_comic_book marvel_comic marvel_cinematic_universe middle_east robertdowneyjr. terrencehoward jeffbridges shauntoub gwynethpaltrow jonfavreau Action Adventure Science_Fiction'

In [104]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

count = CountVectorizer(stop_words='english')
count_matrix = normalize(count.fit_transform(soup_df['soup']), norm = 'l2')
cosine_sim_count = cosine_similarity(count_matrix, count_matrix)

In [105]:
count_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63335 stored elements and shape (4803, 16883)>

In [106]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(soup_df['soup'])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [107]:
def get_recommendations(title, cosine_sim):
    idx = indices.loc[title].values[0]
    sim_scores = enumerate(cosine_sim[idx])

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies_df['title'].iloc[movie_indices]

def get_recommendations_multi_input(movies, cosine_sim):
    sim_scores = []
    input_movie_indices = []
    for title in movies:
        idx = indices.loc[title].values[0]
        input_movie_indices.append(idx)
        sim_scores.append(cosine_sim[idx])

    weights = [3, 2, 1]

    sim_scores = np.average(np.array(sim_scores), axis = 0, weights = weights)
    
    sim_scores = enumerate(normalize(sim_scores.reshape(1,-1), norm='l2').flatten())

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    movie_indices = [i[0] for i in sim_scores if i[0] not in input_movie_indices][:10]

    # return movie_indices

    return indices.iloc[movie_indices].reset_index()['title']

In [114]:
movie = "The Pianist"

display(
    # get_recommendations(movie, cosine_sim_count),
    get_recommendations(movie, cosine_sim_tfidf)
    )


1818        Schindler's List
24                 King Kong
456                     Fury
889        The Thin Red Line
4329              Casablanca
571     Inglourious Basterds
586        The Monuments Men
642                 Unbroken
3197          Flame & Citron
3750        The Great Escape
Name: title, dtype: object

In [ ]:
movies = [
            'Housefull',
            'Babel',
            'Iron Man',
]

display(
# get_recommendations_multi_input(movies, cosine_sim_count),
get_recommendations_multi_input(movies, cosine_sim_tfidf)
)

In [ ]:
indices = movies_df[['title', 'overview']].reset_index().set_index('title')

In [ ]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)

cosine_sim_path = '../artifacts/cosine_sim.pkl'
indices_path = '../artifacts/indices.pkl'

joblib.dump(cosine_sim_tfidf.astype('float32'), cosine_sim_path)
joblib.dump(indices, indices_path)